# Exercise 3: TTS intelligibility and a listening-test design

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Speech Technologies · version 0.1.0*

> Compare TTS voices with an ASR-based intelligibility proxy, measure how much repeated sampling of the same text changes it, and design a blinded, counterbalanced listening test for fluent, compensated listeners. Obtain permission for any cloned voice.

**Setup.** `pip install sherpa-onnx==1.13.8 librosa==0.11.0 soundfile requests numpy`. Everything runs on the CPU without a GPU or the Hugging Face Hub:

- **ASR:** OpenAI Whisper *small*, as the int8 ONNX export published in the sherpa-onnx GitHub release `asr-models` (639 MB archive), decoded greedily by sherpa-onnx.
- **TTS:** the Piper VITS voice `kk_KZ-issai-high` (six speakers, trained from scratch on the ISSAI KazakhTTS data, CC BY 4.0 according to its model card) and, in Exercise 4, `tr_TR-dfki-medium` (one speaker, CC BY-NC-SA 4.0), from the sherpa-onnx release `tts-models`.
- **Text:** the Kazakh and Turkish sides of the X-WMT English–Kazakh and English–Turkish test sets in the commit-pinned TIL release (MIT licence), with the spaces around punctuation removed.

Archives are checked against the SHA-256 digests in the code and cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), together with every synthesised waveform and every transcript, so a rerun reuses them. **The test speech is synthetic.** It comes from a TTS model trained on read studio speech; its error rates say nothing about Whisper on real Kazakh recordings, and they are used here only to compare conditions on identical input. Piper samples its output (noise scale 0.667), so a fresh cache produces slightly different audio and slightly different numbers.

The six voices of `kk_KZ-issai-high` read the same 20 sentences. Whisper’s character error rate measures the combined synthesis-and-recognition pipeline. It can identify examples to inspect, but cannot establish human intelligibility or naturalness. The last part generates the trial lists for a listening test.

In [ ]:
import collections, hashlib, json, os, re, statistics, tarfile, time, unicodedata
from pathlib import Path
import numpy as np, requests

TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
SHERPA = "https://github.com/k2-fsa/sherpa-onnx/releases/download/"
ARCHIVES = {   # sherpa-onnx release archive -> SHA-256 of the file used for the book's run
    "asr-models/sherpa-onnx-whisper-small.tar.bz2": "486a46afbb7ba798507190ffe02fea2dd726049af212e774537efac6afb210a6",
    "tts-models/vits-piper-kk_KZ-issai-high.tar.bz2": "6c59955f7f5e3fc50b130ced59a430eb4ef62d69155af4ce836db0dde4a42682",
    "tts-models/vits-piper-tr_TR-dfki-medium.tar.bz2": "b63babaefcdd63202c7079aba25b72a2013825ad5895e2aeff03d1ae5a5b3bbf",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)
SR = 16000                                     # Whisper's sampling rate
THREADS = os.cpu_count() or 1

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(url, timeout=600, stream=True) as r:
            r.raise_for_status()
            with open(str(path) + ".part", "wb") as f:
                for block in r.iter_content(1 << 20):
                    f.write(block)
        os.replace(str(path) + ".part", path)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    FETCHED[url] = h.hexdigest()
    return path

def archive(name):
    """Download, verify and unpack a sherpa-onnx release archive; return the unpacked folder."""
    path = fetch(SHERPA + name)
    if FETCHED[SHERPA + name] != ARCHIVES[name]:
        raise ValueError(f"{name}: the release file has changed; check it before using it")
    root = path.parent / path.name.removesuffix(".tar.bz2")
    if not root.exists():
        with tarfile.open(path) as t:
            t.extractall(path.parent, filter="data")
    return root

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation)."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt_side(pair, lang):
    """One side of an X-WMT test set, detokenised."""
    text = fetch(f"{TIL}xwmt/test/{pair}/{pair}.{lang}").read_text(encoding="utf-8").splitlines()
    return [detok(s) for s in text if s.strip()]

def test_sentences(pair, lang, n):
    """The first n sentences with 5-20 words and no digits (number expansion is a separate problem)."""
    return [s for s in xwmt_side(pair, lang) if not re.search(r"\d", s) and 5 <= len(s.split()) <= 20][:n]

# ---- TTS: Piper VITS voices through sherpa-onnx; every waveform is cached on disk -------------
VOICES = {"kk": "kk_KZ-issai-high", "tr": "tr_TR-dfki-medium"}
_TTS = {}

def tts(voice, noise=None):
    if (voice, noise) not in _TTS:
        import sherpa_onnx
        root = archive(f"tts-models/vits-piper-{voice}.tar.bz2")
        vits = sherpa_onnx.OfflineTtsVitsModelConfig(model=str(root / f"{voice}.onnx"), tokens=str(root / "tokens.txt"),
                                                     data_dir=str(root / "espeak-ng-data"))
        if noise is not None:                   # noise=0 makes the voice deterministic
            vits.noise_scale = vits.noise_scale_w = noise
        _TTS[voice, noise] = sherpa_onnx.OfflineTts(sherpa_onnx.OfflineTtsConfig(
            model=sherpa_onnx.OfflineTtsModelConfig(vits=vits, num_threads=THREADS)))
    return _TTS[voice, noise]

def speech(text, sid=0, lang="kk", draw=0, noise=None):
    """16 kHz float32 synthetic speech for text; `draw` indexes repeated samples of the same input."""
    key = hashlib.sha1(json.dumps([VOICES[lang], sid, draw, noise, text], ensure_ascii=False).encode()).hexdigest()
    path = CACHE / "ch11-audio" / f"{key}.npy"
    if not path.exists():
        import librosa
        a = tts(VOICES[lang], noise).generate(text, sid=sid, speed=1.0)
        w = librosa.resample(np.asarray(a.samples, dtype=np.float32), orig_sr=a.sample_rate, target_sr=SR)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.save(path, w.astype(np.float32))
    return np.load(path)

# ---- ASR: Whisper small (int8 ONNX), greedy; every transcript is cached ----------------------
_ASR = {}
ASR_CACHE = CACHE / "ch11-transcripts.jsonl"

def _asr(lang):
    if lang not in _ASR:
        import sherpa_onnx
        root = archive("asr-models/sherpa-onnx-whisper-small.tar.bz2")
        _ASR[lang] = sherpa_onnx.OfflineRecognizer.from_whisper(
            encoder=str(root / "small-encoder.int8.onnx"), decoder=str(root / "small-decoder.int8.onnx"),
            tokens=str(root / "small-tokens.txt"), language=lang, task="transcribe", num_threads=THREADS)
    return _ASR[lang]

def _transcripts():
    out = {}
    if ASR_CACHE.exists():
        for line in ASR_CACHE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[r["key"]] = r
    return out
TRANSCRIPTS = _transcripts()

def transcribe(wave, lang="kk"):
    """(text, language Whisper used, decoding seconds); lang="" lets Whisper detect the language."""
    wave = np.ascontiguousarray(wave, dtype=np.float32)
    key = hashlib.sha1(wave.tobytes()).hexdigest() + "|" + lang
    if key not in TRANSCRIPTS:
        rec = _asr(lang)
        s = rec.create_stream(); s.accept_waveform(SR, wave)
        t0 = time.time(); rec.decode_stream(s)
        r = {"key": key, "text": s.result.text.strip(), "lang": s.result.lang, "sec": time.time() - t0}
        TRANSCRIPTS[key] = r
        ASR_CACHE.parent.mkdir(parents=True, exist_ok=True)
        with ASR_CACHE.open("a", encoding="utf-8") as f:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    r = TRANSCRIPTS[key]
    return r["text"], r["lang"], r["sec"]

# ---- Scoring: the same functions as code/remaining/speech.py -----------------------------------
def normalize(text):
    # Conservative scoring: preserve letters, diacritics and apostrophes.
    text = unicodedata.normalize("NFC", text).lower()
    return " ".join("".join(c if not unicodedata.category(c).startswith("P") or c in "'ʻʼ’" else " " for c in text).split())

def distance(a, b):
    row = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        nxt = [i]
        for j, y in enumerate(b, 1):
            nxt.append(min(nxt[-1] + 1, row[j] + 1, row[j - 1] + (x != y)))
        row = nxt
    return row[-1]

def errors(refs, hyps):
    """Corpus-level raw and normalised WER and CER (edits / reference words or characters)."""
    if len(refs) != len(hyps) or not refs:
        raise ValueError("unaligned/empty references")
    result = {}
    for name, fn in [("raw", lambda x: x), ("normalized", normalize)]:
        pairs = [(fn(r), fn(h)) for r, h in zip(refs, hyps)]
        for unit, split in [("wer", str.split), ("cer", list)]:
            denom = sum(len(split(r)) for r, h in pairs)
            edits = sum(distance(split(r), split(h)) for r, h in pairs)
            result[f"{name}_{unit}"] = edits / denom if denom else None
        result[f"{name}_empty_references"] = sum(not r for r, h in pairs)
    return result

def pct(x):
    return f"{100 * x:5.1f}%"

def audio_digest(waves):
    h = hashlib.sha256()
    for w in waves:
        h.update(np.ascontiguousarray(w, dtype=np.float32).tobytes())
    return h.hexdigest()[:16]

print("archives verified:", ", ".join(n.rsplit("/", 1)[-1] for n in ARCHIVES if archive(n)))

In [ ]:
kk_all = xwmt_side("en-kk", "kk")
TEST = test_sentences("en-kk", "kk", 60)               # 60 sentences, spoken by the six voices in turn
waves = [speech(s, sid=i % 6) for i, s in enumerate(TEST)]
hyps, langs, secs = zip(*(transcribe(w, "kk") for w in waves))
audio = sum(len(w) for w in waves) / SR
print(f"X-WMT en-kk test: {len(kk_all)} Kazakh sentences; the first {len(TEST)} with 5-20 words and no digits are used")
print(f"synthetic audio: {audio / 60:.1f} min, {min(len(w) for w in waves) / SR:.1f}-{max(len(w) for w in waves) / SR:.1f} s per item; audio digest {audio_digest(waves)}")
print(f"decoding: {sum(secs):.0f} s on {THREADS} CPU threads, real-time factor {sum(secs) / audio:.2f}")
r = errors(TEST, list(hyps))
print(f"Whisper small, language token kk, greedy:  raw WER {pct(r['raw_wer'])}  CER {pct(r['raw_cer'])}   "
      f"normalised WER {pct(r['normalized_wer'])}  CER {pct(r['normalized_cer'])}")
short = sum(len(normalize(h)) < 0.5 * len(normalize(r)) for r, h in zip(TEST, hyps))
print(f"hypotheses shorter than half their reference (normalised characters): {short} of {len(TEST)}")
print("normalised CER by voice:")
for sid in range(6):
    idx = [i for i in range(len(TEST)) if i % 6 == sid]
    e = errors([TEST[i] for i in idx], [hyps[i] for i in idx])
    print(f"  speaker {sid}: CER {pct(e['normalized_cer'])}  WER {pct(e['normalized_wer'])}  ({len(idx)} items)")
item = sorted(range(len(TEST)), key=lambda i: distance(list(normalize(TEST[i])), list(normalize(hyps[i]))) / len(normalize(TEST[i])))
for name, i in [("best", item[0]), ("median", item[len(item) // 2]), ("worst", item[-1])]:
    c = distance(list(normalize(TEST[i])), list(normalize(hyps[i]))) / len(normalize(TEST[i]))
    print(f"{name} item (CER {pct(c)}):\n  REF {TEST[i]}\n  HYP {hyps[i]}")

In [ ]:
SENTS = TEST[:20]
print("ASR-based intelligibility proxy: Whisper small (kk) on the same 20 sentences from each Piper voice")
cer = {}
for sid in range(6):
    hyps = [transcribe(speech(s, sid=sid), "kk")[0] for s in SENTS]
    e = errors(SENTS, hyps); cer[sid] = e["normalized_cer"]
    print(f"  speaker {sid}: normalised CER {pct(e['normalized_cer'])}  WER {pct(e['normalized_wer'])}")
print(f"  spread between voices: {pct(max(cer.values()) - min(cer.values()))} CER")

print("Sampling variation: speaker 0, the first 10 sentences, three independent draws and one with noise 0")
for draw in range(3):
    e = errors(SENTS[:10], [transcribe(speech(s, sid=0, draw=draw), "kk")[0] for s in SENTS[:10]])
    print(f"  draw {draw}: CER {pct(e['normalized_cer'])}")
e = errors(SENTS[:10], [transcribe(speech(s, sid=0, noise=0.0), "kk")[0] for s in SENTS[:10]])
print(f"  noise 0 : CER {pct(e['normalized_cer'])}")

# A blinded, counterbalanced listening-test plan: 6 systems (here the six voices), 12 sentences,
# 12 listeners. Listener l hears sentence j from system (j + l) mod 6, so each listener hears every
# sentence once and every system twice, and every sentence-system cell is rated by 2 listeners.
SYSTEMS, ITEMS, LISTENERS = 6, 12, 12
salt = "ch11-listening-test"                           # keep the key file away from the listeners
plan, key = {}, {}
for l in range(LISTENERS):
    trials = [(j, (j + l) % SYSTEMS) for j in range(ITEMS)]
    np.random.default_rng(1000 + l).shuffle(trials)
    plan[l] = []
    for j, s in trials:
        blind = hashlib.sha1(f"{salt}|{j}|{s}".encode()).hexdigest()[:8]
        key[blind] = (j, s); plan[l].append(blind)
per_listener = {collections.Counter(key[b][1] for b in plan[l]).most_common()[-1][1] for l in plan}
cells = collections.Counter(key[b] for l in plan for b in plan[l])
first_pos = collections.Counter(key[plan[l][0]][1] for l in plan)
print(f"listening plan: {LISTENERS} listeners x {ITEMS} trials; each listener hears every system "
      f"{per_listener.pop()} times and every sentence once; each of {len(cells)} cells rated "
      f"{set(cells.values()).pop()} times; systems in first position: {dict(sorted(first_pos.items()))}")
print("listener 0 playlist:", " ".join(plan[0]))

## Worked solution

Output of `experiments/ch11_exercises/run_ch11_exercises.py`:

```
ASR-based intelligibility proxy: Whisper small (kk) on the same 20 sentences from each Piper voice
  speaker 0: normalised CER  55.5%  WER  93.8%
  speaker 1: normalised CER  48.8%  WER  93.8%
  speaker 2: normalised CER  58.5%  WER  95.8%
  speaker 3: normalised CER  53.4%  WER  92.0%
  speaker 4: normalised CER  55.1%  WER  92.7%
  speaker 5: normalised CER  57.3%  WER  96.2%
  spread between voices:   9.7% CER
Sampling variation: speaker 0, the first 10 sentences, three independent draws and one with noise 0
  draw 0: CER  55.4%
  draw 1: CER  56.8%
  draw 2: CER  56.5%
  noise 0 : CER  59.8%
listening plan: 12 listeners x 12 trials; each listener hears every system 2 times and every sentence once; each of 72 cells rated 2 times; systems in first position: {0: 2, 1: 3, 2: 1, 3: 2, 4: 1, 5: 3}
listener 0 playlist: 983371f5 eaa668f0 ef112614 76875930 5f106211 c8e0a1cc 31b72bd9 6543d9af 4c68c4bc 0f3a2669 62b58df5 c0f0f146
```

**The voices differ.** With the same 20 sentences, the proxy CER ranges from 48.8% (speaker 1) to 58.5% (speaker 2), a spread of 9.7 points. Speaker 1 also had the lowest CER in Exercise 1, on different sentences, so the ordering is not only an artefact of the sentence assignment. It is still a statement about how well Whisper small recognises each voice, which mixes the voice's clarity with the recogniser's own preferences; a listening test is needed to establish intelligibility.

**Sampling matters too.** Piper samples durations and latent noise, so the same text synthesised three times gives CERs of 55.4%, 56.8% and 56.5% on ten sentences. Setting the noise scales to zero makes the output deterministic but here *raises* CER to 59.8%. A TTS comparison should therefore fix the synthesis settings, save the exact waveforms that listeners hear, and report how much repeated sampling changes the result.

**The listening plan.** Each of 12 listeners hears 12 sentences, each sentence once and each system (voice) twice, and every sentence–system combination is rated by two listeners, so no listener hears the same sentence twice and every system appears with every sentence. File names are salted hashes, hiding the system labels; audible voice differences remain identifiable. For blinded analysis, the key file stays with the experimenter. The trial order is shuffled per listener, but the count of first positions (1 to 3 per system) shows that random shuffling does not balance order effects exactly; a Williams design or a fixed set of balanced orders does. Add training trials and anchors, recruit fluent speakers of the target variety, pay them, record which variety they speak, and decide the analysis (for example a mixed model with listener and sentence effects) before collecting ratings.

## Your turn — extension

1. Compare `length_scale` values of 0.9, 1.0 and 1.2 (slower speech) with the proxy. Does slower speech help the recogniser?
2. Extend the plan to also collect naturalness ratings, and decide whether the same listeners should give both judgments.
3. Write the consent text you would use for a listener, and the one you would need before cloning a voice.